# Aarya Developer Cloud: Google Colab
Run the FastAPI backend inside this Colab runtime. No Render, Google Cloud Billing, or public tunnel is required. The server is temporary and stops when the runtime disconnects.

## 1. Clone or update the GitHub repository

In [ ]:
from pathlib import Path
import subprocess, sys, platform

REPO_URL = 'https://github.com/Aaryashreyansh/Aarya-Developer-Cloud.git'
REPO_DIR = Path('/content/Aarya-Developer-Cloud')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
%cd /content/Aarya-Developer-Cloud
print('Repository ready:', REPO_DIR)
print('Python:', sys.version)
print('Platform:', platform.platform())

## 2. Install dependencies and run the project tests

In [ ]:
%pip -q install -r requirements.txt
!pytest -q

## 3. Configure this temporary backend
The app token is generated privately in this runtime and is not printed or saved to GitHub. Add your chosen provider key in Colab's **Secrets** panel (key icon), for example `GEMINI_API_KEY` or `OPENAI_API_KEY`, and grant this notebook access. Never paste a key into a code cell.

In [ ]:
import os, secrets
from getpass import getpass

from google.colab import userdata

os.environ['APP_API_TOKEN'] = secrets.token_urlsafe(32)
provider = input('Provider (gemini/openai/claude/openrouter) [gemini]: ').strip().lower() or 'gemini'
settings = {
    'gemini': ('GEMINI_API_KEY', 'GEMINI_MODEL'),
    'openai': ('OPENAI_API_KEY', 'OPENAI_MODEL'),
    'claude': ('ANTHROPIC_API_KEY', 'ANTHROPIC_MODEL'),
    'openrouter': ('OPENROUTER_API_KEY', 'OPENROUTER_MODEL'),
}
if provider not in settings:
    raise ValueError('Choose gemini, openai, claude, or openrouter.')
key_name, model_name = settings[provider]
try:
    os.environ[key_name] = userdata.get(key_name)
except Exception:
    print(f'Optional AI test will be skipped unless {key_name} is added in Colab Secrets.')
model_id = input(f'Model ID for {provider} (leave blank to use server environment default): ').strip()
if model_id:
    os.environ[model_name] = model_id
print('Backend authentication configured. Secret values are not displayed.')

## 4. Start FastAPI inside Colab

In [ ]:
import subprocess, time, httpx, sys, os

if 'server_process' in globals() and server_process.poll() is None:
    server_process.terminate()
server_process = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'app.main:app', '--host', '127.0.0.1', '--port', '8000'],
    cwd=str(REPO_DIR), env=os.environ.copy(),
    stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
)
base_url = 'http://127.0.0.1:8000'
for attempt in range(30):
    if server_process.poll() is not None:
        raise RuntimeError('Backend process stopped. Check dependencies and project logs.')
    try:
        health = httpx.get(f'{base_url}/health', timeout=2)
        if health.status_code == 200:
            break
    except httpx.HTTPError:
        time.sleep(1)
else:
    raise TimeoutError('Backend did not start within 30 seconds.')
print('Backend is running locally:', health.json())
print('API docs:', base_url + '/docs')

## 5. Test the API and provider list

In [ ]:
providers_response = httpx.get(f'{base_url}/api/v1/providers', timeout=10)
providers_response.raise_for_status()
print('Supported providers:', [p['id'] for p in providers_response.json()['providers']])
print('Health check:', httpx.get(f'{base_url}/health').json())

## 6. Optional real AI request

In [ ]:
key_name, model_name = settings[provider]
if not os.getenv(key_name):
    print(f'Skipped: add {key_name} in Colab Secrets to enable real AI requests.')
else:
    payload = {
        'provider': provider,
        'prompt': 'Reply with exactly: Aarya Developer Cloud connection successful.',
        'system_prompt': 'You are a concise connection-test assistant.'
    }
    response = httpx.post(
        f'{base_url}/api/v1/chat',
        headers={'X-API-Token': os.environ['APP_API_TOKEN']},
        json=payload, timeout=90,
    )
    response.raise_for_status()
    result = response.json()
    print('Provider:', result['provider'])
    print('AI response:', result['text'])

## Notes
- This backend runs only inside the current Colab runtime. It is not a permanent public website or 24/7 server.
- Do not print, share, or commit API keys or app tokens. Keep provider keys in Colab Secrets.
- Provider API usage may incur charges according to the provider's pricing and account limits.
- Stop the backend with `server_process.terminate()` when finished.
- The backend returns generated text; it does not execute generated code.